In [1]:
import cv2
import torch
import torchvision.transforms as transforms
from model import CNNModel


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNNModel()
model.load_state_dict(torch.load('best_model2.pth')['model_state_dict'])
model = model.to(device)
model.eval()


C:\Users\otun2\AppData\Local\Temp\ipykernel_23716\3437213231.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model2.pth')['model_s

CNNModel(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1))
  (conv5): Conv2d(512, 1024, kernel_size=(3, 3), stride=(1, 1))
  (pool5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv6): Conv2d(1024, 2048, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=51200, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=7, bias=True)

In [7]:
video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    print("Error: Could not open video.")
    exit()
while True:
    ret, img = video_capture.read()
    if not ret:
        print("Error: Could not read frame.")
        break
    
    # convert BGR image to grayscale image to match original datasett
    gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #using the face cascade model to detect faces
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    #detects faces and returns a list of rectangles, representing a detected face as (x,y,w,h)
    faces = face_cascade.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=5)
    for (x,y,w,h) in faces:
        cv2.rectangle(img,(x,y), (x+w,y+h), (255,0,0),thickness=7)
        roi_gray = gray_image[y:y+h,x:x+w]
    #convert image into tensor so the model can use
        image_resized = cv2.resize(roi_gray, (48,48))

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.4870],std=[0.2284])
        ])
        image_tensor = transform(image_resized)
        image_tensor = image_tensor.unsqueeze(0).to(device)

        output = model(image_tensor)
        _, predicted = torch.max(output,1)
        emotion = predicted.item()
        emotion_dict = {0: "Angry", 1: "Disgust",2:"Fear",3:"Happy",4:"Sad",5:"Surprise",6:"Neutral"}
        emotion_pred = emotion_dict[emotion]
        cv2.putText(img,emotion_pred,(int(x),int(y)), cv2.FONT_HERSHEY_SIMPLEX,2,(0,255,0),3)
    resized_img = cv2.resize(img, (1000, 700))
    cv2.imshow('Emotion', resized_img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
video_capture.release()
cv2.destroyAllWindows()